# Lab: Extracting Raw Training Data from LLMs (The Privacy Attack)

To demonstrate the findings of Carlini et al. in "Extracting Training Data from Large Language Models," we need to simulate their **likelihood-based extraction attack**.

The core idea is that models assign an abnormally high probability to sequences they have seen during training. However, "common" sequences (like "the end of the") also have high probability. To find *memorized* data, Carlini et al. look for sequences where the probability is high, but the "surprise" compared to a generic model is low.

### The Attack Concept

The paper uses three main strategies to find memorized data. We will implement the **Likelihood Ratio** method:

1. **Generate** many samples from a "victim" LLM (e.g., GPT-2).
2. **Score** those samples using the victim's own confidence ($P_\theta$).
3. **Compare** that score against a reference model ($P_{ref}$) or a zlib compression score.
4. **Rank** them. High ratios indicate the model is "over-confident" in that specific string, often signifying memorization.

---

#### Step 1: Setup and Model Loading

We use GPT-2 (Small) as the "Victim." While the paper used GPT-2 (XL) for better results, the Small version still demonstrates the principle on a free GPU.

In [ ]:
# Install dependencies
#!pip install transformers torch zlib-ng

import torch
import zlib
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the victim model (GPT-2 Small) and its tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2" # The "victim"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Model loaded on {device}")

2026-04-26 18:41:50.957231: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

#### Step 2: Generating Candidate Sequences

We generate a batch of text. The paper used various sampling strategies (top-k, temperature); we'll use a simple random sampling to find potential "leaks."

In [ ]:
def generate_samples(num_samples=20, max_length=64):
    samples = []
    # Using a null prompt to let the model ramble
    input_ids = torch.tensor([[tokenizer.bos_token_id]]).to(device)

    print(f"Generating {num_samples} samples...")
    for _ in range(num_samples):
        output = model.generate(
            input_ids,
            do_sample=True,
            max_length=max_length,
            top_k=40,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
        text = tokenizer.decode(output[0], skip_special_tokens=False)
        samples.append(text)
    return samples


In [ ]:
candidates = generate_samples(5)
print("Samples generated.")

#### Step 3: The Ranking Metric (The Attack)

Carlini et al. found that sorting by the ratio of **Log-Likelihood / Zlib Entropy** is a powerful way to find memorized text. Zlib acts as a proxy for how much "generic" information is in the text.

In [ ]:
def calculate_perplexity(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        # Log-likelihood of the sequence
        return outputs.loss.item()

def get_zlib_entropy(text):
    # Compression ratio as a baseline for "natural" text complexity
    return len(zlib.compress(bytes(text, 'utf-8')))



In [ ]:
results = []
for text in candidates:
    if len(text.strip()) < 10: continue

    ll = calculate_perplexity(text)
    zlib_score = get_zlib_entropy(text)

    # Carlini Metric: Ratio of Perplexity to Zlib complexity
    # Lower score usually means more likely to be memorized
    score = ll / zlib_score
    results.append((score, text))

# Sort by the most "suspicious" (lowest ratio)
results.sort(key=lambda x: x[0])

print(f"{'Score':<10} | {'Extracted Text Snippet'}")
print("-" * 50)
for score, text in results[:10]:
    snippet = text.replace('\n', ' ')[:80]
    print(f"{score:<10.4f} | {snippet}...")

---

### Analysis of the Results

1. **Likelihood vs. Memorization**: In your results, you will notice that sequences with very low scores are often repetitive or contain specific "boilerplate" text (licenses, headers, or common internet phrases).
2. **Privacy Implications**: Carlini et al. found that GPT-2 XL had memorized contact information (emails, phone numbers) from the internet. Because GPT-2 Small has fewer parameters, its "capacity" for memorization is lower, but you may still see verbatim snippets of open-source licenses or famous quotes.
3. **Why it works**:
* **Low Perplexity**: The model is extremely "sure" about what comes next.
* **Low Zlib/Reference Ratio**: The model is *more* sure about this specific string than it usually is for text of that complexity.



### How to Verify "Ground Truth"

In the original study, the researchers compared their extracted snippets against the **Common Crawl** dataset to prove they were verbatim training data. For your demo, you can take a low-scoring snippet from your output and search for it on Google (wrapped in quotes) to see if it exists verbatim on the web, this is effectively what the researchers did at scale.